In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cbook as cbook
from matplotlib import image

from sklearn.model_selection import train_test_split

np.random.seed(0)

## 1. Background

Everything we have built so far draws a straight line. Logistic regression computes one weighted
sum $z = \theta^T x$, squashes it with a sigmoid, and the boundary where $P = 0.5$ is always a
hyperplane. To bend it we had to invent features by hand, like $x^2$.

A **neural network** lets the model invent its own features. Each **hidden unit** is its own little
logistic-style unit: a weighted sum followed by a nonlinearity. The output layer is then just a
logistic classifier that uses the hidden units as its features.

Today we will build one from scratch in NumPy, with no deep-learning library, one piece at a time:

1. the **forward pass**: activation functions, softmax, and the network itself

2. the **loss**: cross-entropy

3. **backpropagation**: the gradient of the loss with respect to every weight

4. **stochastic gradient descent**: the training loop

5. **convolution and pooling**: the two operations that turn a network into a CNN

Every function you write gets called by the next one, so none of them can be skipped.

## 2. The Data

We are back with the penguins, but this time the species column is the answer key rather than
something to hide. We have 342 birds, four measurements each, and three classes: Adelie, Chinstrap
and Gentoo. That means our network needs **three outputs**, one probability per species.

In [ ]:
df = pd.read_csv("penguins.csv")

features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
species_names = np.array(sorted(df["species"].unique()))

X = df[features].to_numpy(dtype=float)
y = pd.Categorical(df["species"], categories=species_names).codes.astype(int)   # 0, 1, 2

print("X shape:", X.shape)
print("Classes:", dict(enumerate(species_names)))
print("Counts: ", np.bincount(y))

Two pieces of setup before we touch the network.

**Hold out validation data.** A network can memorize its training set, so we score it on penguins it
never trained on. `stratify=y` keeps the species proportions the same in both splits.

**Standardize.** Remember from Week 7 that body mass is measured in grams and swamps everything
else. The same problem hits gradient descent: weights attached to a huge feature get huge gradients.
We compute the mean and standard deviation on the **training set only**, then apply those same
numbers to the validation set, so that no information leaks from validation into training.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

mu, sigma = X_train.mean(axis=0), X_train.std(axis=0)
X_train = (X_train - mu) / sigma
X_val = (X_val - mu) / sigma

print("Train:", X_train.shape, "  Validation:", X_val.shape)

## 3. The Forward Pass

### 3.1 The activation function

Without a nonlinearity, stacking layers is pointless: $W_2(W_1x) = (W_2W_1)x$, which is just one
linear layer. The modern default for hidden units is the **rectified linear unit**:

$$\text{ReLU}(z) = \max(0, z)$$

It passes positive values through unchanged and clips negatives to exactly zero. It should work on
an array of any shape, element by element.

*Hint: `np.maximum` compares two arrays (or an array and a number) element-wise.*

In [ ]:
def relu(z):
    """
    Element-wise rectified linear unit.

    z:       array of any shape
    returns: array of the same shape, with every negative entry replaced by 0
    """

    return np.maximum(0, z)

#### Let's double check our functions!

In [ ]:
z = np.array([[-2.0, -0.5, 0.0], [0.5, 1.0, 3.0]])
got = relu(z)
expected = np.array([[0.0, 0.0, 0.0], [0.5, 1.0, 3.0]])
assert np.allclose(got, expected), f"Expected {expected}, Actual {got}"

# ReLU must not modify its input in place.
assert z[0, 0] == -2.0, f"Expected -2.0, Actual {z[0, 0]}"

print("Passed")

### 3.2 Softmax

With $K$ classes the output layer produces $K$ raw scores per example, called **logits**. Softmax
turns each row of logits into probabilities that are positive and sum to 1:

$$\text{softmax}(z)_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

There is one trap. `np.exp(1000)` overflows to infinity. The fix is free: subtracting the same
number $c$ from every logit in a row does not change the answer, because the $e^{-c}$ cancels
between the top and bottom. So subtract each row's **maximum** before exponentiating, and the
largest exponent you ever compute is $e^0 = 1$.

The input `Z` has shape `(n, K)`: one row per example, one column per class.

*Hint: `Z.max(axis=1, keepdims=True)` gives each row's max in a shape that broadcasts back over the row.*

In [ ]:
def softmax(Z):
    """
    Row-wise softmax.

    Z:       (n, K) array of logits
    returns: (n, K) array of probabilities; every row is positive and sums to 1
    """

    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)

#### Let's double check our functions!

In [ ]:
Z = np.array([[1.0, 2.0, 3.0], [0.0, 0.0, 0.0]])
got = softmax(Z)
expected = np.array([[0.09003057, 0.24472847, 0.66524096], [1/3, 1/3, 1/3]])
assert np.allclose(got, expected), f"Expected {expected}, Actual {got}"

# Every row sums to 1.
assert np.allclose(got.sum(axis=1), 1.0), f"Expected rows summing to 1, Actual {got.sum(axis=1)}"

# Huge logits must not overflow into nan.
big = softmax(np.array([[1000.0, 1001.0, 1002.0]]))
assert np.allclose(big, expected[0]), f"Expected {expected[0]}, Actual {big}"

print("Passed")

### 3.3 The network

Now the whole forward pass. Our network has one hidden layer with $M$ units:

$$H = \text{ReLU}(XW_1 + b_1) \qquad P = \text{softmax}(HW_2 + b_2)$$

| Name | Shape | Meaning |
|---|---|---|
| `X` | `(n, d)` | a batch of $n$ examples, $d$ features each |
| `W1`, `b1` | `(d, M)`, `(M,)` | input → hidden weights and biases |
| `H` | `(n, M)` | hidden unit values, the **learned features** |
| `W2`, `b2` | `(M, K)`, `(K,)` | hidden → output weights and biases |
| `P` | `(n, K)` | predicted probability of each class |

*Why is this $XW_1$ when the slides wrote $Wx$?* The slides handled one example as a column vector.
Here each example is a **row** of `X`, so the whole batch goes through in one matrix product and
`W1` is the transpose of the slides' $W$. Same network.

Backpropagation will need the intermediate values later, so `forward` also returns a `cache`
dictionary holding `Z1` (the hidden layer's weighted sums, before ReLU) and `H` (after ReLU).

*Hint: this is four lines, and it should call `relu` and `softmax`.*

In [ ]:
def forward(X, params):
    """
    Forward pass through a one-hidden-layer network.

    X:       (n, d) array of inputs
    params:  dict with "W1" (d, M), "b1" (M,), "W2" (M, K), "b2" (K,)
    returns: P, cache
             P     -- (n, K) array of class probabilities
             cache -- dict with "Z1" (n, M) pre-activation values and "H" (n, M) hidden values
    """

    Z1 = X @ params["W1"] + params["b1"]
    H = relu(Z1)
    P = softmax(H @ params["W2"] + params["b2"])
    return P, {"Z1": Z1, "H": H}

#### Let's double check our functions!

This is the forward pass we did by hand on the slides: input $x = (1, 2)$, hidden unit 1 has weights
$(0.5, -1)$ and bias $1$, hidden unit 2 has weights $(1, 0.5)$ and bias $-1$, and the output weights
are $(2, -1)$ with bias $0.5$.

The slides used a single sigmoid output. A softmax over two logits $[0, a]$ gives exactly
$\sigma(a)$ for the second class, so we give class 0 all-zero weights and put the slide's weights on
class 1. We should get $h = (0, 1)$ and $P(\text{class } 1) = \sigma(-0.5) \approx 0.378$.

In [ ]:
slide_params = {
    "W1": np.array([[0.5, 1.0],
                    [-1.0, 0.5]]),          # column j holds hidden unit j's weights
    "b1": np.array([1.0, -1.0]),
    "W2": np.array([[0.0, 2.0],
                    [0.0, -1.0]]),          # class 0 gets zeros; class 1 gets the slide's (2, -1)
    "b2": np.array([0.0, 0.5]),
}
P, cache = forward(np.array([[1.0, 2.0]]), slide_params)

assert np.allclose(cache["Z1"], [[-0.5, 1.0]]), f"Expected [[-0.5, 1.0]], Actual {cache['Z1']}"
assert np.allclose(cache["H"], [[0.0, 1.0]]), f"Expected [[0.0, 1.0]], Actual {cache['H']}"
expected = 1 / (1 + np.exp(0.5))
assert np.isclose(P[0, 1], expected), f"Expected {expected:.4f}, Actual {P[0, 1]:.4f}"
assert P.shape == (1, 2), f"Expected shape (1, 2), Actual {P.shape}"

print("Passed")

### 3.4 Starting weights

The network needs somewhere to start. Two rules matter here.

* **The weights must be random.** If every hidden unit starts with identical weights, they compute
identical outputs, receive identical gradients, and stay identical forever. We will prove that in
section 7.

* **The scale matters.** Weights that are too large blow the activations up; too small and the
signal fades. For ReLU layers a standard choice is a normal distribution with variance $2/d$
(**He initialization**). We start the output weights very small, so that the untrained network
begins by guessing all three species about equally.

In [ ]:
def init_params(d, M, K, seed=0):
    """Random starting weights for a d -> M -> K network."""
    rng = np.random.default_rng(seed)
    return {
        "W1": rng.normal(0, np.sqrt(2 / d), size=(d, M)),
        "b1": np.zeros(M),
        "W2": rng.normal(0, 0.01, size=(M, K)),
        "b2": np.zeros(K),
    }

params = init_params(d=4, M=16, K=3)
P, _ = forward(X_train[:5], params)
print("Untrained predictions for 5 penguins:")
print(P.round(3))

Roughly one third each. The network knows nothing yet, which is exactly where we want it to start.

## 4. The Loss

We score the predictions with **cross-entropy**, the same loss as logistic regression: the average
negative log-probability the model gave to the **correct** class.

$$L = \frac{1}{n}\sum_{i=1}^{n} -\log P_{i,\,y_i}$$

If the model puts probability 1 on the right species, that term is $-\log 1 = 0$. If it puts
probability near 0 there, the term shoots toward infinity.

*Hint: `P[np.arange(n), y]` picks out one entry per row: row `i`, column `y[i]`.*

In [ ]:
def cross_entropy(P, y):
    """
    Mean cross-entropy loss.

    P:       (n, K) array of predicted probabilities
    y:       (n,) array of integer labels in [0, K)
    returns: a single float
    """

    n = len(y)
    return -np.mean(np.log(P[np.arange(n), y]))

#### Let's double check our functions!

In [ ]:
# Perfect, confident predictions cost nothing.
P_perfect = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
got = cross_entropy(P_perfect, np.array([0, 1]))
assert np.isclose(got, 0.0), f"Expected 0.0, Actual {got}"

# Guessing uniformly over 3 classes costs log(3).
P_uniform = np.full((4, 3), 1 / 3)
got = cross_entropy(P_uniform, np.array([0, 1, 2, 0]))
assert np.isclose(got, np.log(3)), f"Expected {np.log(3):.4f}, Actual {got:.4f}"

# Mixed case: -(log 0.5 + log 0.25) / 2
got = cross_entropy(np.array([[0.5, 0.5, 0.0], [0.25, 0.5, 0.25]]), np.array([0, 2]))
expected = -(np.log(0.5) + np.log(0.25)) / 2
assert np.isclose(got, expected), f"Expected {expected:.4f}, Actual {got:.4f}"

print("Passed")

That uniform case gives us a free sanity check. A network that knows nothing should score about
$\log 3 \approx 1.099$. Let's see whether ours does.

In [ ]:
P, _ = forward(X_train, params)
print(f"Untrained loss: {cross_entropy(P, y_train):.4f}")
print(f"log(3):         {np.log(3):.4f}")

## 5. Backpropagation

To train, we need $\partial L / \partial w$ for every weight. Backpropagation gets all of them with
the chain rule, walking from the loss back toward the input and reusing what it already computed.

**Step 1: the output error.** For softmax combined with cross-entropy, the gradient with respect to
the logits simplifies beautifully:

$$\frac{\partial L}{\partial Z_2} = \frac{1}{n}\,(P - Y)$$

where $Y$ is the **one-hot** version of `y` (a 1 in the true class's column, 0 elsewhere). This is the
multi-class version of the $(f - y)$ from the slides: prediction minus truth.

**Step 2: output layer weights.** Each weight's gradient is the error times whatever flowed into it:

$$\frac{\partial L}{\partial W_2} = H^T \frac{\partial L}{\partial Z_2} \qquad
\frac{\partial L}{\partial b_2} = \text{sum over rows of } \frac{\partial L}{\partial Z_2}$$

**Step 3: send the error back through $W_2$ and through ReLU.** ReLU's slope is 1 where $Z_1 > 0$
and 0 elsewhere, so units that were switched off pass no gradient back:

$$\frac{\partial L}{\partial H} = \frac{\partial L}{\partial Z_2} W_2^T \qquad
\frac{\partial L}{\partial Z_1} = \frac{\partial L}{\partial H} \odot [Z_1 > 0]$$

**Step 4: first layer weights.** Same pattern as step 2:

$$\frac{\partial L}{\partial W_1} = X^T \frac{\partial L}{\partial Z_1} \qquad
\frac{\partial L}{\partial b_1} = \text{sum over rows of } \frac{\partial L}{\partial Z_1}$$

*Hint: build `dZ2` by copying `P`, subtracting 1 at `[np.arange(n), y]`, then dividing by `n`. Every
gradient must have exactly the same shape as the parameter it belongs to.*

In [ ]:
def backward(X, y, params, P, cache):
    """
    Gradients of the mean cross-entropy loss with respect to every parameter.

    X:       (n, d) inputs used in the forward pass
    y:       (n,) integer labels
    params:  dict with "W1", "b1", "W2", "b2"
    P:       (n, K) probabilities returned by forward
    cache:   dict with "Z1" and "H" returned by forward
    returns: dict with "W1", "b1", "W2", "b2", each the same shape as the matching parameter
    """

    n = len(y)
    dZ2 = P.copy()
    dZ2[np.arange(n), y] -= 1
    dZ2 /= n

    grads = {"W2": cache["H"].T @ dZ2, "b2": dZ2.sum(axis=0)}

    dH = dZ2 @ params["W2"].T
    dZ1 = dH * (cache["Z1"] > 0)

    grads["W1"] = X.T @ dZ1
    grads["b1"] = dZ1.sum(axis=0)
    return grads

#### Let's double check our functions!

How do we know a gradient is right? We can **measure** it. Nudge one weight up by a tiny $\epsilon$,
nudge it down, and see how much the loss moves:

$$\frac{\partial L}{\partial w} \approx \frac{L(w + \epsilon) - L(w - \epsilon)}{2\epsilon}$$

This is far too slow for training (two forward passes per weight), but it is the standard way to
test a backprop implementation. If the two agree to many decimal places, the calculus is right.

In [ ]:
def numerical_gradient(X, y, params, name, eps=1e-6):
    """Estimate dL/d(params[name]) by nudging each entry up and down."""
    grad = np.zeros_like(params[name])
    for idx in np.ndindex(params[name].shape):
        original = params[name][idx]
        params[name][idx] = original + eps
        loss_up = cross_entropy(forward(X, params)[0], y)
        params[name][idx] = original - eps
        loss_down = cross_entropy(forward(X, params)[0], y)
        params[name][idx] = original
        grad[idx] = (loss_up - loss_down) / (2 * eps)
    return grad

check_params = init_params(d=4, M=5, K=3, seed=1)
check_params["W2"] *= 100            # bigger output weights so every gradient is clearly nonzero
X_small, y_small = X_train[:10], y_train[:10]

P, cache = forward(X_small, check_params)
grads = backward(X_small, y_small, check_params, P, cache)

for name in ["W1", "b1", "W2", "b2"]:
    assert grads[name].shape == check_params[name].shape, \
        f"Expected shape {check_params[name].shape}, Actual {grads[name].shape}"
    numeric = numerical_gradient(X_small, y_small, check_params, name)
    error = np.max(np.abs(grads[name] - numeric))
    assert error < 1e-6, f"Expected {name} gradient error below 1e-6, Actual {error:.2e}"
    print(f"{name}: max difference from numerical gradient = {error:.1e}")

print("Passed")

## 6. Training with Stochastic Gradient Descent

Full gradient descent computes the gradient over **every** training example before taking one step.
**Stochastic** gradient descent takes a step after every small **minibatch** of $b$ examples:

1. Shuffle the training data.

2. Walk through it $b$ examples at a time. For each minibatch: forward pass, record the loss,
backward pass, then update every parameter with $\theta \leftarrow \theta - \eta \nabla L$.

One pass through the whole dataset is one **epoch**. With $n$ examples that is
$\lceil n / b \rceil$ updates, where the last minibatch may be smaller than $b$.

Update `params` **in place** (e.g. `params["W1"] -= lr * grads["W1"]`), so the caller sees the
trained weights. Return the list of minibatch losses.

*Hint: `rng.permutation(n)` gives a shuffled order of the indices `0 … n-1`; slice it in steps of
`batch_size`.*

In [ ]:
def sgd_epoch(X, y, params, lr, batch_size, rng):
    """
    Run one epoch of minibatch stochastic gradient descent, updating params in place.

    X, y:        training inputs (n, d) and integer labels (n,)
    params:      dict of parameters, modified in place
    lr:          learning rate
    batch_size:  examples per gradient step
    rng:         a numpy Generator, used to shuffle
    returns:     list of the loss on each minibatch, in the order they were processed
    """

    order = rng.permutation(len(y))
    losses = []
    for start in range(0, len(y), batch_size):
        batch = order[start:start + batch_size]
        P, cache = forward(X[batch], params)
        losses.append(cross_entropy(P, y[batch]))
        grads = backward(X[batch], y[batch], params, P, cache)
        for name in params:
            params[name] -= lr * grads[name]
    return losses

#### Let's double check our functions!

In [ ]:
test_params = init_params(d=4, M=8, K=3, seed=2)
before = cross_entropy(forward(X_train, test_params)[0], y_train)

losses = sgd_epoch(X_train, y_train, test_params, lr=0.1, batch_size=32, rng=np.random.default_rng(0))
after = cross_entropy(forward(X_train, test_params)[0], y_train)

expected_updates = int(np.ceil(len(y_train) / 32))
assert len(losses) == expected_updates, f"Expected {expected_updates}, Actual {len(losses)}"
assert after < before, f"Expected loss to drop below {before:.4f}, Actual {after:.4f}"

# A batch size of n is ordinary full-batch gradient descent: exactly one update.
one = sgd_epoch(X_train, y_train, init_params(4, 8, 3), lr=0.1, batch_size=len(y_train),
                rng=np.random.default_rng(0))
assert len(one) == 1, f"Expected 1, Actual {len(one)}"

print(f"Loss before one epoch: {before:.4f}   after: {after:.4f}")
print("Passed")

### 6.1 Training the network

Everything is in place. The loop below just calls `sgd_epoch` repeatedly and records the loss on
the training and validation sets after each epoch.

In [ ]:
def train(M=16, lr=0.1, batch_size=32, epochs=30, seed=0):
    """Train a fresh 4 -> M -> 3 network. Returns (params, train_losses, val_losses)."""
    params = init_params(d=4, M=M, K=3, seed=seed)
    rng = np.random.default_rng(seed)
    train_losses, val_losses = [], []
    for _ in range(epochs):
        sgd_epoch(X_train, y_train, params, lr, batch_size, rng)
        train_losses.append(cross_entropy(forward(X_train, params)[0], y_train))
        val_losses.append(cross_entropy(forward(X_val, params)[0], y_val))
    return params, train_losses, val_losses


def accuracy(X, y, params):
    """Fraction of examples whose most probable class is the true class."""
    P, _ = forward(X, params)
    return np.mean(P.argmax(axis=1) == y)


params, train_losses, val_losses = train()

plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="training loss")
plt.plot(val_losses, label="validation loss")
plt.xlabel("epoch")
plt.ylabel("cross-entropy")
plt.title("Training a network we wrote ourselves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Training accuracy:   {accuracy(X_train, y_train, params):.3f}")
print(f"Validation accuracy: {accuracy(X_val, y_val, params):.3f}")

Let's see which penguins it still gets wrong.

In [ ]:
val_pred = forward(X_val, params)[0].argmax(axis=1)
print(pd.crosstab(pd.Series(species_names[y_val], name="true species"),
                  pd.Series(species_names[val_pred], name="predicted")))

Any mistakes will be between Adelie and Chinstrap. Just as in Week 7, those two species overlap on
these measurements while Gentoo sits apart, so the network faces the same hard boundary the
clustering did. This time it has labels to learn from.

### 6.2 Batch size

On the slides, smaller batches made much more progress for the same number of passes through the
data. Let's check that on our own network: same learning rate, same 30 epochs, three batch sizes.

In [ ]:
n_train = len(y_train)
plt.figure(figsize=(7, 4))
for b in [n_train, 32, 8]:
    _, tr, _ = train(batch_size=b)
    label = f"b = {b}" + ("  (full-batch GD)" if b == n_train else "")
    updates = int(np.ceil(n_train / b))
    plt.plot(tr, label=f"{label}: {updates} update{'s' if updates > 1 else ''} per epoch")
plt.xlabel("epoch")
plt.ylabel("training cross-entropy")
plt.title("Same epochs, different batch sizes")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Every curve saw each penguin exactly 30 times. Full-batch gradient descent turned that into 30
careful steps; $b = 8$ turned it into 900 noisy ones and got much further. That is the whole
argument for SGD, and it gets stronger as datasets grow.

## 7. Why Initialization Matters

Section 3.4 claimed that hidden units must start with **different** weights. Let's test it. We will
start all 16 hidden units with the same weights, train exactly as before, and then look at what they
learned.

In [ ]:
same_start = {
    "W1": np.full((4, 16), 0.5),     # every hidden unit starts identical
    "b1": np.zeros(16),
    "W2": np.full((16, 3), 0.01),
    "b2": np.zeros(3),
}
rng = np.random.default_rng(0)
for _ in range(30):
    sgd_epoch(X_train, y_train, same_start, lr=0.1, batch_size=32, rng=rng)

spread = np.abs(same_start["W1"] - same_start["W1"][:, [0]]).max()
print("Largest difference between any two hidden units' weights:", spread)
print(f"Validation accuracy: {accuracy(X_val, y_val, same_start):.3f}")

A difference of exactly zero. All 16 units received identical gradients at every step, so they
never separated, and our 16-unit network is really a **1-unit** network wearing a disguise. That is
why every real initializer is random: randomness is what lets the units specialize into different
features.

## 8. Convolution and Pooling

A dense layer connects every hidden unit to every input. For images that explodes: a 1000 × 1000
colour image has 3 million inputs, so every single hidden unit would need 3 million weights.

A **convolutional** layer uses a small **filter** (say 3 × 3) instead, and slides the same filter
across the whole image. At each position it multiplies the filter element-wise with the patch
underneath and sums the result. A $k \times k$ filter fits into an $H \times W$ image at
$(H - k + 1) \times (W - k + 1)$ positions, so that is the output shape.

### 8.1 Convolution

*Hint: two nested loops over output positions `(r, c)`. The patch is `image[r:r+k, c:c+k]`.*

(Strictly, this operation is called cross-correlation, because the filter is not flipped. Every
deep-learning library does it this way and still calls it convolution.)

In [ ]:
def conv2d(image, kernel):
    """
    Slide kernel over image (no padding, stride 1), taking an element-wise product and sum at each position.

    image:   (H, W) array
    kernel:  (k, k) array
    returns: (H - k + 1, W - k + 1) array of filter responses
    """

    k = kernel.shape[0]
    out_h, out_w = image.shape[0] - k + 1, image.shape[1] - k + 1
    out = np.zeros((out_h, out_w))
    for r in range(out_h):
        for c in range(out_w):
            out[r, c] = np.sum(image[r:r + k, c:c + k] * kernel)
    return out

#### Let's double check our functions!

This is the 6 × 6 example from the slides, with the diagonal filter. The top-left response should be
$3$, and the full 4 × 4 output should match the slide exactly.

In [ ]:
slide_image = np.array([[1, 0, 0, 0, 0, 1],
                        [0, 1, 0, 0, 1, 0],
                        [0, 0, 1, 1, 0, 0],
                        [1, 0, 0, 0, 1, 0],
                        [0, 1, 0, 0, 1, 0],
                        [0, 0, 1, 0, 1, 0]], dtype=float)
diagonal_filter = np.array([[1, -1, -1],
                            [-1, 1, -1],
                            [-1, -1, 1]], dtype=float)

got = conv2d(slide_image, diagonal_filter)
expected = np.array([[3, -1, -3, -1],
                     [-3, 1, 0, -3],
                     [-3, -3, 0, 1],
                     [3, -2, -2, -1]], dtype=float)
assert got.shape == (4, 4), f"Expected shape (4, 4), Actual {got.shape}"
assert np.allclose(got, expected), f"Expected {expected}, Actual {got}"

print("Passed")

### 8.2 Max pooling

**Pooling** shrinks a feature map by keeping only the largest value in each block. With 2 × 2 blocks
it halves the height and width. It keeps the answer to "did this filter fire somewhere in this
region?" and throws away exactly where, which buys some tolerance to small shifts. It has no
weights to learn.

You can assume the height and width divide evenly by `size`.

*Hint: loop over block corners in steps of `size`, or reshape to `(H/size, size, W/size, size)`
and take the max over axes 1 and 3.*

In [ ]:
def max_pool(fmap, size=2):
    """
    Non-overlapping max pooling.

    fmap:    (H, W) array, with H and W divisible by size
    size:    block edge length
    returns: (H // size, W // size) array holding the max of each block
    """

    H, W = fmap.shape
    return fmap.reshape(H // size, size, W // size, size).max(axis=(1, 3))

#### Let's double check our functions!

In [ ]:
got = max_pool(conv2d(slide_image, diagonal_filter))   # the 4 x 4 response from the slide
want = np.array([[3.0, 0.0], [3.0, 1.0]])
assert np.allclose(got, want), f"Expected {want}, Actual {got}"

got = max_pool(np.arange(16.0).reshape(4, 4))
want = np.array([[5.0, 7.0], [13.0, 15.0]])
assert np.allclose(got, want), f"Expected {want}, Actual {got}"

print("Passed")

### 8.3 One convolutional layer on a real image

A convolutional layer is exactly the three functions you have written, applied in order:
**convolve, ReLU, pool**. Let's run it on a real photo with two hand-designed edge filters, the same
ones from the slides.

In a real CNN nobody designs these filters. Their weights are learned by the backprop and SGD you
just wrote, and trained networks tend to rediscover edge detectors much like these in their first
layer. (The loops in `conv2d` take a few seconds on an image this size; libraries do the same
computation with fast matrix operations.)

In [ ]:
with cbook.get_sample_data("grace_hopper.jpg") as f:
    photo = image.imread(f).mean(axis=2) / 255.0      # colour -> grayscale in [0, 1]
photo = photo[::2, ::2]                               # downsample so the loops stay quick

vertical_edges = np.array([[1, 0, -1],
                           [2, 0, -2],
                           [1, 0, -1]], dtype=float)
horizontal_edges = vertical_edges.T

def conv_layer(img, kernel):
    return max_pool(relu(conv2d(img, kernel)))

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
axes[0].imshow(photo, cmap="gray")
axes[0].set_title(f"input {photo.shape}")
for ax, kernel, name in zip(axes[1:], [vertical_edges, horizontal_edges], ["vertical", "horizontal"]):
    out = conv_layer(photo, kernel)
    ax.imshow(out, cmap="gray", vmax=np.percentile(out, 99))     # clip the brightest 1% so edges are visible
    ax.set_title(f"{name}-edge channel {out.shape}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

Each filter produced its own **channel**, lit up wherever its pattern appears, at half the
resolution thanks to pooling. Stack many channels and repeat the block a few times, and you have the
feature extractor at the front of every image classifier. Count the parameters too: each filter here
has **9** weights, no matter how large the photo is.

## 9. On Your Own

Section 3.1 claimed that without a nonlinearity, depth buys nothing. Test it. The helper below
trains a network with **either** ReLU or no activation at all (the identity), on any dataset.
Run it on the penguins and on the two moons from the slides.

In [ ]:
from sklearn.datasets import make_moons

def fit(X_tr, y_tr, X_te, y_te, use_relu=True, M=16, lr=0.1, epochs=300, batch_size=32, seed=0):
    '''Train a d -> M -> K network with or without ReLU. Returns validation accuracy.'''
    act = relu if use_relu else (lambda z: z)
    p = init_params(d=X_tr.shape[1], M=M, K=int(y_tr.max()) + 1, seed=seed)
    rng = np.random.default_rng(seed)
    def fwd(X):
        Z1 = X @ p["W1"] + p["b1"]
        H = act(Z1)
        return softmax(H @ p["W2"] + p["b2"]), {"Z1": Z1, "H": H}
    for _ in range(epochs):
        order = rng.permutation(len(y_tr))
        for start in range(0, len(y_tr), batch_size):
            b = order[start:start + batch_size]
            P, cache = fwd(X_tr[b])
            if not use_relu:
                cache = {"Z1": np.ones_like(cache["Z1"]), "H": cache["H"]}   # identity: slope 1 everywhere
            grads = backward(X_tr[b], y_tr[b], p, P, cache)
            for name in p:
                p[name] -= lr * grads[name]
    return np.mean(fwd(X_te)[0].argmax(axis=1) == y_te)

X_m, y_m = make_moons(n_samples=400, noise=0.2, random_state=0)
Xm_tr, Xm_va, ym_tr, ym_va = train_test_split(X_m, y_m, test_size=0.3, random_state=0)

for name, data in [("Penguins", (X_train, y_train, X_val, y_val)), ("Two moons", (Xm_tr, ym_tr, Xm_va, ym_va))]:
    lin = fit(*data, use_relu=False)
    nonlin = fit(*data, use_relu=True)
    print(f"{name:10s}  no activation: {lin:.3f}   ReLU: {nonlin:.3f}")

On the penguins the two are about even, and the linear network may even edge ahead. That is not a
contradiction: the three species are close to **linearly separable** on these measurements, so a
straight boundary is already enough and extra flexibility has nothing to buy. On the moons, whose
classes need a curved boundary, the identity network is stuck with a line and ReLU pulls clearly
ahead. A nonlinearity only pays off when the problem is nonlinear.

*(How does `fit` reuse your `backward` for the identity? `backward` multiplies by the mask
`Z1 > 0`; handing it a cache where every `Z1` is positive makes that mask all ones, which is the
identity's slope.)*

Now experiment. Change `M` to 1 or 256, or `lr` to 5.0 or 0.001, in either `fit` or `train()`.
Which change hurts the most, and can you explain it using the training and validation curves?

## 10. Recap

* A **neural network** is logistic regression that learns its own features. Each hidden unit is a
weighted sum followed by a **nonlinearity**, and without that nonlinearity every layer collapses
into one linear model.

* **Softmax** turns logits into class probabilities; subtracting the row max keeps it from
overflowing. **Cross-entropy** scores those probabilities, and an untrained network should score
about $\log K$.

* **Backpropagation** is the chain rule, run from the output back to the input, reusing the values
saved in the forward pass. For softmax plus cross-entropy the output error is simply $P - Y$.
A **numerical gradient check** is how you know it is right.

* **SGD** trades exact gradients for many cheap, noisy ones. For the same number of epochs, smaller
batches make far more progress.

* **Initialization** must be random. Identical hidden units get identical gradients and never
separate.

* A **convolutional layer** is convolve, ReLU, pool: one small filter reused at every position,
with a parameter count that does not depend on the image size.